<p align="center">
<a href="https://duckietown.com"><img src="../assets/images/dtlogo.png" alt="Duckietown Logo" width="50%"></a>
</p>


# Subscribing to ToF

The downward __time-of-flight (ToF)__ sensor publishes range readings through a driver and ROS 2 bridge.

This notebook writes the node at the end of that chain. The node subscribes to the downward sensor's topic and prints the newest reading every second, so the value can be watched as the distance under the sensor changes.

Only the downward sensor runs by default, so this notebook uses that one.

Use only a Duckiedrone you are explicitly authorized to access. For a physical Duckiedrone, obtain permission for the hardware, credentials, and network path. For a virtual Duckiedrone, start the Duckiematrix and attach an entity you are authorized to operate before expecting sensor data.

By the time a reading reaches the topic, the [driver](./2-duckiedrone-tof-architecture.ipynb#the-driver) and the [bridge](./2-duckiedrone-tof-architecture.ipynb#the-ros-2-bridge) have already converted it to $\mathrm{m}$, stamped it, and marked unusable readings. The node only has to ask ROS 2 for the messages and unpack them.

By the end of the notebook you should be able to answer:

- Which program publishes the downward ToF topic, and which programs subscribe to it?

- How often does the downward ToF topic publish, and how can that be checked?

- What does one printed line describe?

- Which printed values are not distances, and what should a consumer do with them?

## Getting to a terminal that can see the Duckiedrone

Use a Duckiedrone shell to inspect the sensor services.

### Open a shell on the Duckiedrone

On a physical Duckiedrone:

```bash
ssh duckie@DUCKIEDRONE_NAME.local
```

For a virtual Duckiedrone:

```bash
dts duckiebot virtual connect DUCKIEDRONE_NAME
```

### Find the containers that publish the ToF

```bash
docker ps --format '{{.Names}}'
```

The standard downward-ToF pair is `driver-tof-bottom` and `ros2-tof-bottom`. The driver reads the sensor, while the bridge publishes its ROS 2 topic.

### Open a shell inside the bridge

```bash
docker exec -it ros2-tof-bottom bash
```

On a virtual Duckiedrone, simulated readings arrive only while the Duckiematrix is running and the Duckiedrone is attached to an entity.

## Look at the data first

Inspect the ToF-specific data path before writing a line of code.

First, list the ToF-related topics:

```bash
ros2 topic list | grep tof
```

On the example physical Duckiedrone `pdrone24`, the topic list includes:

```shell
/pdrone24/bottom_tof_driver_node/range
```

This is the topic from [Notebook 2](./2-duckiedrone-tof-architecture.ipynb#the-ros-2-bridge), with the Duckiedrone's hostname in front.

Next, inspect its publisher and subscribers:

```bash
ros2 topic info /DUCKIEDRONE_NAME/bottom_tof_driver_node/range
```

```shell
Type: sensor_msgs/msg/Range
Publisher count: 1
Subscription count: 3
```

The publisher is the bridge. The subscribers are MAVROS2, which forwards the readings to the flight controller, and two other Duckiedrone nodes that read the range, `altitude_node` and `state_estimator_node`.

Then measure the publication rate:

```bash
ros2 topic hz /DUCKIEDRONE_NAME/bottom_tof_driver_node/range
```

```shell
average rate: 30.001
        min: 0.021s max: 0.044s std dev: 0.00265s window: 798
```

The downward sensor is configured for $30\,\mathrm{Hz}$, as [Table 3 of Notebook 2](./2-duckiedrone-tof-architecture.ipynb#table-3) shows, and this output from a physical Duckiedrone matches. A virtual Duckiedrone can differ, so measure it rather than assume. Ctrl-C stops the command.

Finally, print one message:

```bash
ros2 topic echo --once /DUCKIEDRONE_NAME/bottom_tof_driver_node/range
```

In what `echo` prints, `range` is the reported distance in $\mathrm{m}$, `header.stamp` is when the driver read the measurement, and `header.frame_id` identifies the sensor. Compare those fields with [Table 2 of Notebook 2](./2-duckiedrone-tof-architecture.ipynb#table-2). The node written next uses all three.

## Activity 1: write the subscriber

<a id="activity-1"></a>

Open [packages/solution/solution/tof_subscriber.py](../packages/solution/solution/tof_subscriber.py). The starter declares the parameters, builds each full topic name, keeps `self.latest_readings`, and starts a timer that calls `cb_log`. Fill the two `TO IMPLEMENT` blocks.

### In `__init__`, create the subscription

For each `full_topic`, create a `Range` subscription with `self._make_callback(name)` and `qos_profile_sensor_data`, then append the returned subscription to `self.subs`. The ToF bridge publishes with that sensor-data QoS profile; inspect it with:

```bash
ros2 topic info -v /DUCKIEDRONE_NAME/bottom_tof_driver_node/range
```

<a id="unpack-one-message"></a>

### In `_make_callback`, unpack one message

The callback made for each `name` stores that sensor's newest `ToFReading`. Set `range_m` from `float(msg.range)`, `stamp_sec` from the seconds and nanoseconds in `msg.header.stamp`, and `frame_id` from `msg.header.frame_id`. A virtual Duckiedrone can leave `frame_id` empty, which is why the callback keeps the sensor name passed to `_make_callback`.

Store the reading rather than logging it from the callback. `cb_log` prints the newest stored reading on its timer.

With both pieces written, the node is ready to run.

## Activity 2: run the node on the Duckiedrone

<a id="activity-2"></a>

From the project's root folder, build and run the node:

```bash
dts code build -R DUCKIEDRONE_NAME
dts code workbench -R DUCKIEDRONE_NAME
```

The node's output appears in the terminal:

```shell
[tof_listener_node-1] [INFO] [tof_listener_node]: Subscribing to /pdrone24/bottom_tof_driver_node/range  (sensor: bottom)
[tof_listener_node-1] [INFO] [tof_listener_node]: ranges=[bottom=0.018m]
```

The first line identifies the subscribed topic. Its hostname and sensor name should match the topic inspected above. After changing the code, repeat the same build-and-workbench sequence.

### When nothing arrives

If the node keeps printing `Waiting for ToF data...`, first check that both `TO IMPLEMENT` blocks from [Activity 1](#activity-1) are complete. Without the subscription code, the node logs its topic but never subscribes.

For this sensor path, check these sensor-specific points:

- The topic logged at startup must have one publisher in `ros2 topic info`.

- `ros2 topic hz /DUCKIEDRONE_NAME/bottom_tof_driver_node/range` should report about $30\,\mathrm{Hz}$ on a
  physical Duckiedrone. On a virtual Duckiedrone, also confirm that the Duckiematrix is running and
  the Duckiedrone is attached.

- `docker ps` should list `driver-tof-bottom` and `ros2-tof-bottom`; if either is missing, nothing
  publishes the topic.

## Activity 3: make the number move

<a id="activity-3"></a>

Each printed line is the newest reading, unchanged. With the node running, change what the downward sensor sees: on a physical Duckiedrone, slide a book slowly under it and take it away; on a virtual one, move the Duckiedrone up and down with the Duckiematrix controls.

Three kinds of reading are worth catching:

- __Below the minimum.__ A physical Duckiedrone resting on its legs prints something like
  `0.018m`, below the message's `min_range` of `0.04`. The sensor is only specified down to
  $4\,\mathrm{cm}$, but it still reports what it measured, and neither the driver nor the bridge
  clamps the value. A virtual Duckiedrone on the ground reads about `0.047m`, so this case appears
  only on a physical one.

- __In range.__ $20\,\mathrm{cm}$ or $30\,\mathrm{cm}$: the ordinary case, and the one reading of the
  three that is a usable distance.

- __No reading.__ With the motors off, hold the Duckiedrone so the downward sensor faces open
  space with nothing within $3.6\,\mathrm{m}$, for example down a corridor. The line does not go
  blank: it prints `999.000m`, the value the bridge uses when the driver has no distance to report.

Two of the three are not usable distances, yet the node prints all three the same way. That is fine for watching the data, but anything that acts on the number has to discard values outside `min_range` and `max_range` first, as the [message definition](https://github.com/ros2/common_interfaces/blob/jazzy/sensor_msgs/msg/Range.msg) asks.

## Going further

Each of these is optional, and each one builds on the node from [Activity 1](#activity-1):

- Discard what the message definition says to discard. Copy `min_range` and `max_range` into
  `ToFReading`, and have `cb_log` print `out of range` instead of the number when `range` falls
  outside them.

- Watch every value rather than once per second: `ros2 topic echo
  /DUCKIEDRONE_NAME/bottom_tof_driver_node/range --field range`.

## Further reading

The [sensor_msgs/Range message definition](https://github.com/ros2/common_interfaces/blob/jazzy/sensor_msgs/msg/Range.msg) lists the fields and range validity rules used by the subscriber.

## Checkpoint

Run the self-check in the next cell. Write or select a response before revealing the answer.


In [ ]:
import sys
from pathlib import Path

working_directory = Path.cwd()
parent_directory = working_directory.parent
if (parent_directory / "packages").is_dir():
    parent_directory_path = str(parent_directory)
    sys.path.insert(0, parent_directory_path)

from packages.checkpoint_self_check import display_checkpoint_self_checks

display_checkpoint_self_checks()